# Description of NS Data Downloaded (Dutch, refer to site for English version)

### Train Disruptions
Als je deze dataset gebruikt is het belangrijk om te beseffen dat deze data gaat over verstoringen die door NS gecommuniceerd zijn. Niet elke trein die vertraagd of opgeheven is wordt door NS gecommuniceerd als een verstoring; de vuistregel die NS hanteert is dat een verstoring gecommuniceerd wordt wanneer meerdere treinen vertraagd of opgeheven zijn (dus een grote impact op de treindienst).

Het is ook belangrijk om te beseffen dat er sinds 2017 meer verstoringen gecommuniceerd worden, omdat NS toen een nieuw systeem heeft geïntroduceerd waarmee ze verstoringen eerder kunnen aankondigen (wat resulteert in meer verstoringen met een korte duur). Het vergelijken van het aantal verstoringen van 2017 met het aantal verstoringen in de jaren daarvoor is daardoor niet mogelijk (tenzij je rekening houdt met de toename van korte verstoringen).

De bron voor de verstoringen is altijd NS; de afdeling reisinformatie v
an NS monitort 24 uur per dag de treindienst om te zien of er verstoringen zijn. De storingsberichten in de open data zijn dezelfde berichten als op de borden op het station, de stationsomroep en op de Rijden de Treinen website en app.

https://www.rijdendetreinen.nl/open-data/treinstoringen

### Train Services 
TODO: check if we need this as we have disruptions already, 
note that previous theses seem to use this services dataset instead of disruptions dataset

Alle treinritten in Nederland worden sinds 2019 opgeslagen in het treinarchief van Rijden de Treinen. Deze dataset bevat alle vertragingen, opgeheven treinen en dienstregelingswijzigingen van alle treinen in Nederland.

Deze dataset bevat alle reizigerstreinen in Nederland sinds 2019. De data wordt aangeboden als CSV-bestanden gecomprimeerd met Gzip.

Iedere rij in deze bestanden representeert een stop op een station. Iedere rit vertrekt vanaf en komt aan op een station (dus twee rijen). Voor iedere stop vind je de naam van het station, de aankomst- en vertrektijd, vertragingen en opgeheven ritten. De exacte betekenis van iedere kolom wordt hieronder uitgelegd.

De bron voor deze data is de real-time data van NS met live vertrektijden, actuele aankomsttijden en dienstregelingswijzigingen. Deze data wordt ook gebruikt in de app en website van Rijden de Treinen.

https://www.rijdendetreinen.nl/open-data/treinarchief

### Train Stations
De stations in deze dataset worden gebruikt door de Rijden de Treinen website en app. De bron voor deze data is NS; de stations worden direct uit de NS API gehaald. De stationsnamen, geocoördinaten en stationcodes worden bepaald door NS.

https://www.rijdendetreinen.nl/open-data/treinstations


### Train Tariffs Distances

Deze dataset is een grote matrix met alle treinstations in Nederland (die je kunt herkennen aan de stationscode die te vinden is in de dataset met treinstations). De matrix bevat de afstand van de kortste route tussen ieder station. Er zijn ook tabellen die de routes beperken tot een specifieke vervoerder, wat betekent dat de kortste route langer kan zijn of dat er helemaal geen route beschikbaar is.

De volgende conventies worden gebruikt in de dataset:

Getal: de afstand tussen twee stations.
? (vraagteken): er is geen route beschikbaar tussen de twee stations.
XXX: de twee stations zijn gelijk.

https://www.rijdendetreinen.nl/open-data/tariefafstanden


url = https://opendata.rijdendetreinen.nl/public/stations/stations-2023-09.csv

# Imports

In [1]:
# --- Setup: imports & basic configuration ---
import re
import requests
from bs4 import BeautifulSoup
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

# Optional: progress bar (install if missing)
try:
    from tqdm import tqdm
except ImportError:
    tqdm = None


# NS Open Data URL's

In [12]:
import re
import requests
from bs4 import BeautifulSoup
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

try:
    from tqdm import tqdm
except ImportError:
    tqdm = None

# ✅ Dataset base URLs (now specific per type)
base_url = "https://opendata.rijdendetreinen.nl/public"
page_urls = {
    "disruptions": f"{base_url}/disruptions/",
    "services": f"{base_url}/services/",
    "stations": f"{base_url}/stations/",
    "tariff-distances": f"{base_url}/tariff-distances/"
}

# Output folders
base_dir = Path("downloads/data")
folders = {
    "disruptions": base_dir / "NS-disruptions",
    "services": base_dir / "NS-services",
    "stations": base_dir / "NS-stations",
    "tariff-distances": base_dir / "NS-tariff-distances"
}
for folder in folders.values():
    folder.mkdir(parents=True, exist_ok=True)

## Scrape download links

In [14]:
def get_dataset_links(page_urls):
    links = {key: [] for key in folders.keys()}

    for key, url in page_urls.items():
        try:
            print(f"Fetching links from: {url}")
            r = requests.get(url, timeout=20)
            r.raise_for_status()
            soup = BeautifulSoup(r.text, "html.parser")
            anchors = soup.find_all("a")

            for a in anchors:
                href = a.get("href")
                if not href:
                    continue
                href_abs = requests.compat.urljoin(url, href)

                # Match by dataset type
                if key in ["disruptions", "services"]:
                    m = FILE_RE.search(href_abs)
                    if m:
                        dataset_type, year = m.group(1).lower(), int(m.group(2))
                        if dataset_type == key and year >= 2019:
                            links[key].append(href_abs)

                elif key == "stations" and STATIONS_RE.search(href_abs):
                    links["stations"].append(href_abs)

                elif key == "tariff-distances" and TARIFF_RE.search(href_abs):
                    links["tariff-distances"].append(href_abs)

        except Exception as e:
            print(f"Error fetching {url}: {e}")

    # Sort and deduplicate
    for key in links:
        links[key] = sorted(set(links[key]))

    return links

# Fetch all datasets
links = get_dataset_links(page_urls)

# Summary
for key, urls in links.items():
    print(f"{key}: {len(urls)} files")
    for u in urls:
        print("   ", u)


Fetching links from: https://opendata.rijdendetreinen.nl/public/disruptions/
Fetching links from: https://opendata.rijdendetreinen.nl/public/services/
Fetching links from: https://opendata.rijdendetreinen.nl/public/stations/
Fetching links from: https://opendata.rijdendetreinen.nl/public/tariff-distances/
disruptions: 6 files
    https://opendata.rijdendetreinen.nl/public/disruptions/disruptions-2019.csv
    https://opendata.rijdendetreinen.nl/public/disruptions/disruptions-2020.csv
    https://opendata.rijdendetreinen.nl/public/disruptions/disruptions-2021.csv
    https://opendata.rijdendetreinen.nl/public/disruptions/disruptions-2022.csv
    https://opendata.rijdendetreinen.nl/public/disruptions/disruptions-2023.csv
    https://opendata.rijdendetreinen.nl/public/disruptions/disruptions-2024.csv
services: 6 files
    https://opendata.rijdendetreinen.nl/public/services/services-2019.csv.gz
    https://opendata.rijdendetreinen.nl/public/services/services-2020.csv.gz
    https://opendata

In [15]:
def download_file(url, base_dir):
    filename = url.split("/")[-1].lower()

    # Detect folder by keyword
    if "disruptions" in filename:
        out_path = base_dir / "NS-disruptions" / filename
    elif "services" in filename:
        out_path = base_dir / "NS-services" / filename
    elif "stations" in filename:
        out_path = base_dir / "NS-stations" / filename
    elif "tariff-distances" in filename:
        out_path = base_dir / "NS-tariff-distances" / filename
    else:
        out_path = base_dir / filename  # fallback

    if out_path.exists():
        return url, "exists", out_path

    try:
        with requests.get(url, stream=True, timeout=30) as r:
            r.raise_for_status()
            with open(out_path, "wb") as f:
                for chunk in r.iter_content(8192):
                    f.write(chunk)
        return url, "ok", out_path
    except Exception as e:
        return url, f"error: {e}", out_path


## Download files (2min)

In [16]:
# Flatten all links
all_links = sum(links.values(), [])
print(f"Total files to download: {len(all_links)}")

results = []
with ThreadPoolExecutor(max_workers=6) as ex:
    futs = [ex.submit(download_file, url, base_dir) for url in all_links]
    if tqdm:
        for f in tqdm(as_completed(futs), total=len(futs)):
            results.append(f.result())
    else:
        for f in as_completed(futs):
            results.append(f.result())

# Show summary
for url, status, path in results:
    print(f"{status:10} -> {path}")


Total files to download: 14


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 14/14 [01:43<00:00,  7.41s/it]

ok         -> downloads\data\NS-disruptions\disruptions-2021.csv
ok         -> downloads\data\NS-disruptions\disruptions-2023.csv
ok         -> downloads\data\NS-disruptions\disruptions-2019.csv
ok         -> downloads\data\NS-disruptions\disruptions-2020.csv
ok         -> downloads\data\NS-disruptions\disruptions-2024.csv
ok         -> downloads\data\NS-disruptions\disruptions-2022.csv
ok         -> downloads\data\NS-services\services-2022.csv.gz
ok         -> downloads\data\NS-stations\stations-2023-09-nl.csv
ok         -> downloads\data\NS-tariff-distances\tariff-distances-2022-01.csv
ok         -> downloads\data\NS-services\services-2023.csv.gz
ok         -> downloads\data\NS-services\services-2019.csv.gz
ok         -> downloads\data\NS-services\services-2021.csv.gz
ok         -> downloads\data\NS-services\services-2024.csv.gz
ok         -> downloads\data\NS-services\services-2020.csv.gz


In [17]:
import gzip, shutil

services_dir = folders["services"]

for gz_file in services_dir.glob("*.gz"):
    csv_file = gz_file.with_suffix('')  # remove .gz extension
    if not csv_file.exists():
        with gzip.open(gz_file, 'rb') as f_in:
            with open(csv_file, 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
        print("Extracted:", csv_file)


Extracted: downloads\data\NS-services\services-2019.csv
Extracted: downloads\data\NS-services\services-2020.csv
Extracted: downloads\data\NS-services\services-2021.csv
Extracted: downloads\data\NS-services\services-2022.csv
Extracted: downloads\data\NS-services\services-2023.csv
Extracted: downloads\data\NS-services\services-2024.csv
